In [30]:
# 安裝處理 Word
!pip install python-docx ipywidgets

# 安裝模糊比對
!pip install rapidfuzz

# 確保 Excel 套件為最新
!pip install -U openpyxl
!pip install pandas python-docx openpyxl

In [31]:
from google.colab import files
uploaded_excel = files.upload()
uploaded_word = files.upload()

Saving application_content2.xlsx to application_content2.xlsx


Saving application_corp.docx to application_corp.docx


# 基本資料

In [32]:
from docx import Document
from docx.oxml.ns import qn
import pandas as pd

def get_grid_span(cell):
    grid_span = cell._tc.xpath('./w:tcPr/w:gridSpan')
    if grid_span:
        return int(grid_span[0].get(qn('w:val')))
    return 1

def find_cell_by_visual_col(row, target_visual_col):
    visual_col = 0
    for cell in row.cells:
        span = get_grid_span(cell)
        if visual_col == target_visual_col:
            return cell
        elif visual_col > target_visual_col:
            return None
        visual_col += span
    return None

def set_checkbox_by_keyword(table, keyword, check_val):
    for row in table.rows:
        for c_idx, cell in enumerate(row.cells):
            if keyword in cell.text:
                try:
                    # 假設：「框框 是」「是」「框框 否」「否」依序排列
                    box_yes = row.cells[c_idx]     # 框框 是
                    box_no = row.cells[c_idx + 2]  # 框框 否
                    if check_val == 0:
                        box_yes.text = '☐'
                        box_no.text = '■'
                    else:
                        box_yes.text = '■'
                        box_no.text = '☐'
                    return True
                except:
                    return False
    return False

def fill_cell_by_keyword(table, keyword, value, fill_position='右側'):
    for row_idx, row in enumerate(table.rows):
        visual_col = 0
        for cell in row.cells:
            span = get_grid_span(cell)
            if keyword in cell.text.strip():
                if fill_position == '右側':
                    right_visual_col = visual_col + span
                    right_cell = find_cell_by_visual_col(row, right_visual_col)
                    if right_cell:
                        old_text = right_cell.text.strip()
                        right_cell.text = f"{old_text} {value}" if old_text else str(value)
                        return True
                    else:
                        print(f"⚠️ 關鍵字「{keyword}」右側無儲存格，已略過填寫")
                        return False

                elif fill_position == '下方':
                    next_row_idx = row_idx + 1
                    if next_row_idx < len(table.rows):
                        next_row = table.rows[next_row_idx]
                        down_cell = find_cell_by_visual_col(next_row, visual_col)
                        if down_cell:
                            old_text = down_cell.text.strip()
                            down_cell.text = f"{old_text} {value}" if old_text else str(value)
                            return True
                        else:
                            print(f"⚠️ 關鍵字「{keyword}」下方無儲存格，已略過填寫")
                            return False
                    else:
                        print(f"⚠️ 關鍵字「{keyword}」已在最後一列，無下方儲存格")
                        return False
            visual_col += span
    return False

# === 主程式 ===
excel_path = 'application_content2.xlsx'
ws_data = pd.read_excel(excel_path, sheet_name=0)
ws_map = pd.read_excel(excel_path, sheet_name=3)

data_dict = ws_data.set_index('Excel 欄位名稱')['值'].to_dict()
merged = ws_map.merge(ws_data, on='Excel 欄位名稱', how='left')

doc = Document('application_corp.docx')
table = doc.tables[0]
for idx, row in merged.iterrows():
    keyword = str(row['Word 關鍵字']).strip()
    value = row['值']
    if pd.isna(value) or keyword == '':
        continue

    if keyword in ['僑外投資事業', '一人公司', '陸資']:
        try:
            check_val = int(value)
        except:
            check_val = 0
        success = set_checkbox_by_keyword(table, keyword, check_val)
        if success:
            print(f"✅ 已設定勾選框框 -> 表格0 「{keyword}」")
        else:
            print(f"⚠️ 設定勾選框框失敗 -> 表格0 「{keyword}」")
    else:
        position = str(row.get('填寫位置', '右側')).strip()
        success = fill_cell_by_keyword(table, keyword, value, fill_position=position)
        if success:
            print(f"✅ 已填入：『{value}』 -> 表格0 「{keyword}」 {position}")
        else:
            print(f"⚠️ 無法填入：表格0 「{keyword}」 {position}")

doc.save('基本資料.docx')
print("✅ 基本資料已填寫完成，儲存為基本資料.docx")

/usr/local/lib/python3.11/dist-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


✅ 已填入：『開發科技股份有限公司』 -> 表格0 「中文」 右側
✅ 已填入：『Development Tech Co., Ltd.』 -> 表格0 「(章程所訂)外文」 右側
⚠️ 無法填入：表格0 「(郵遞區號)公司所在地
(含鄉鎮市區村里)」 右側
✅ 已填入：『02-2345-6789』 -> 表格0 「公司聯絡電話」 右側
✅ 已填入：『12345678』 -> 表格0 「公司統一編號」 右側
✅ 已填入：『A20250626001』 -> 表格0 「公司預查編號」 右側
✅ 已填入：『3』 -> 表格0 「董事人數」 右側
✅ 已填入：『張三』 -> 表格0 「代表人姓」 右側
✅ 已填入：『2025-06-20 00:00:00』 -> 表格0 「公司章程訂定日期」 右側
✅ 已填入：『5000000』 -> 表格0 「1.現金」 右側
✅ 已填入：『3000000』 -> 表格0 「2.財產」 右側
✅ 已填入：『2000000』 -> 表格0 「3.技術」 右側
✅ 已填入：『3000000』 -> 表格0 「4.合併新設」 右側
✅ 已填入：『2025-08-01 00:00:00』 -> 表格0 「合併基準日」 下方
✅ 已填入：『98765432』 -> 表格0 「統一編號」 下方
✅ 已填入：『合併公司股份有限公司』 -> 表格0 「公司名稱」 下方
✅ 已設定勾選框框 -> 表格0 「僑外投資事業」
✅ 已設定勾選框框 -> 表格0 「一人公司」
✅ 已設定勾選框框 -> 表格0 「陸資」
✅ 已填入：『10000000』 -> 表格0 「資本總額」 右側
✅ 基本資料已填寫完成，儲存為基本資料.docx


# 所營事業

In [33]:
from docx import Document
import pandas as pd

# 載入 Excel 資料
excel_path = 'application_content2.xlsx'
df_business = pd.read_excel(excel_path, sheet_name='營業項目表')

# 載入 Word 文件並取出第 4 張表格（索引從 0 開始）
doc = Document('application_corp.docx')
table = doc.tables[3]

# 從第 3 列開始填寫（假設前兩列是表頭）
start_row = 2

for i, row in df_business.iterrows():
    code = str(row['Excel 欄位名稱']).strip()
    desc = str(row['營業項目']).strip()

    # 確保表格有足夠的列
    if len(table.rows) <= start_row + i:
        table.add_row()

    table_row = table.rows[start_row + i]
    table_row.cells[1].text = code    # 第 2 欄（索引1）填代碼
    table_row.cells[2].text = desc    # 第 3 欄（索引2）填說明

# 儲存文件
doc.save('所營項目.docx')
print("✅ 所營項目已填寫完成，儲存為所營項目.docx")

✅ 所營項目已填寫完成，儲存為所營項目.docx


# 董事、股東名單

In [ ]:
from docx import Document
import pandas as pd

excel_path = 'application_content2.xlsx'
df = pd.read_excel(excel_path, sheet_name='董事股東名單表')

doc = Document('application_corp.docx')
table = doc.tables[4]  # 第五張表格

start_row = 2  # 從第三列開始填（假設前兩列是表頭）

for i, row in df.iterrows():
    needed_rows = start_row + i * 2 + 1  # 每筆兩列資料
    while len(table.rows) <= needed_rows:
        table.add_row()

    # 第一列：職稱、姓名、身分證號、出資額
    first_row = table.rows[start_row + i * 2]
    first_row.cells[1].text = str(row['Excel 欄位名稱']).strip()  # 職稱
    first_row.cells[2].text = str(row['姓名']).strip()
    first_row.cells[3].text = str(row['身分證號']).strip()
    first_row.cells[4].text = str(row['出資額']).strip()

    # 第二列：地址
    second_row = table.rows[start_row + i * 2 + 1]
    second_row.cells[0].text = str(row['地址']).strip()
    # 其餘欄位清空避免殘留
    for cell in second_row.cells[1:]:
        cell.text = ''

doc.save('董事股東名單.docx')
print("✅ 董事股東名單已填寫完成，儲存為董事股東名單.docx")


✅ 董事股東名單已填寫完成，儲存為董事股東名單.docx
